
## Gaussian Extended Source — Fibre Coupling Efficiency at the Pupil Plane

### 1. Aim

Reproduce the Gaussian extended-source coupling calculation by comparing **coherent and incoherent illumination** of a single guided fibre mode.

Rather than calculating the coupling efficiency at the fibre front face, we perform the modal overlap directly at the **telescope pupil plane**, using the back-propagated fibre mode.

This is possible because a unitary Fourier transform preserves inner products (Parseval's theorem). Therefore, the coupling calculation at the pupil plane is mathematically equivalent to the overlap calculation at the fibre front face, provided both fields use consistent coordinate grids and Fourier-transform conventions.

### 2. Gaussian Surface Brightness Profile

We represent the extended source as a collection of point-source elements at different angular positions on the sky, with a circularly symmetric Gaussian surface brightness profile:

$$
I_{\mathrm{source}}(q_x,q_y)
=
\exp\left[-\frac{q_x^2+q_y^2}{2\sigma^2}\right]
$$

where:

- $q_x$, $q_y$ are angular positions expressed in units of $\lambda/D$.
- $\sigma$ (`source_sigma`) is the Gaussian standard deviation, also in units of $\lambda/D$.
- $\lambda$ is the observing wavelength.
- $D$ is the telescope primary mirror diameter.

The Gaussian is normalised so that:

$$
\sum_{i,j} I_{\mathrm{source},ij}=1.
$$

Therefore, `I_source_sky[iy, ix]` represents the fractional source flux contributed by each angular position.

### 3. Angular Phase Tilt at the Telescope Pupil

Each point-source element at a different angular position on the sky produces a different **linear phase gradient across the telescope pupil**.

A tilt in the pupil plane corresponds to a spatial shift of the point-spread function (PSF) at the focal plane, according to the Fourier shift theorem.

The pupil-plane phase tilt for a point source at angular position $(\theta_x,\theta_y)$ is:

$$
T(X,Y)
=
\exp\left[
\frac{2\pi i}{\lambda}
(\theta_x X+\theta_y Y)
\right].
$$

Since the angular positions are expressed in units of $\lambda/D$,

$$
\theta_x=q_x\frac{\lambda}{D},
\qquad
\theta_y=q_y\frac{\lambda}{D},
$$

we obtain:

$$
\boxed{
T(X,Y)
=
\exp\left[
\frac{2\pi i}{D}(q_xX+q_yY)
\right]
}
$$

In Python:

```python
tilt = np.exp(
    2j * np.pi *
    (QX[iy, ix] * PX + QY[iy, ix] * PY) / D_M1
)
```

Here, `PX` and `PY` are physical coordinates at the telescope pupil in metres.

The phase tilt represents the geometrical phase variation introduced by the point source's angular position, **not the intrinsic phase of the source itself**.

### 4. Point-Source Electric Field at the Pupil Plane

`E_base_pupil` represents the diffraction-limited VLT pupil electric field for an on-axis point source, including the central obstruction.

The pupil electric field from an off-axis point source is:

$$
E_{\mathrm{psf,pupil}}(X,Y)
=
E_{\mathrm{base,pupil}}(X,Y)T(X,Y).
$$

Each source position therefore has its own complex pupil electric field.

### 5. Overlap with the Back-Propagated Fibre Mode

`E_m_pupil` is the electric field of the guided fibre mode, originally calculated at the fibre front face and Fourier-transformed back to the telescope pupil plane.

For each point-source position, we calculate the complex modal overlap:

$$
a_m(q_x,q_y)
=
\left\langle
E_{m,\mathrm{pupil}},
E_{\mathrm{psf,pupil}}
\right\rangle.
$$

In Python, `np.vdot(E_m_pupil, E_psf_pupil)` evaluates the discrete inner product, conjugating the first field.

The point-source modal coupling efficiency is:

$$
\eta_m(q_x,q_y)=|a_m(q_x,q_y)|^2,
$$

provided both the point-source pupil field and the back-propagated fibre mode are normalised to unit power.

### 6. Coherent and Incoherent Extended Sources

The main difference between coherent and incoherent sources is how contributions from different angular positions are combined.

**Incoherent extended source**

For an incoherent source, the relative phases between different source positions are uncorrelated. The time-averaged interference terms vanish, so we add the brightness-weighted modal **powers**:

$$
\boxed{
\eta_{\mathrm{incoherent}}
=
\sum_{i,j} I_{\mathrm{source},ij}
|a_{m,ij}|^2
}
$$

Each source position contributes independently to the total coupled power.

**Coherent extended source**

For a hypothetical fully coherent source, the relative phases are fixed. We assume zero intrinsic phase difference between source elements.

Since intensity is proportional to the squared electric-field amplitude, the amplitude weight of each source element is:

$$
w_{ij}=\sqrt{I_{\mathrm{source},ij}}.
$$

The total coherent pupil electric field is:

$$
E_{\mathrm{source,pupil}}
=
\sum_{i,j}
w_{ij}E_{\mathrm{psf,pupil},ij}.
$$

The complex modal coupling amplitudes must be added **before taking the squared magnitude**:

$$
a_{\mathrm{coherent}}
=
\sum_{i,j}w_{ij}a_{m,ij}.
$$

The total incident coherent power is:

$$
P_{\mathrm{coherent,pupil}}
=
\left\langle
E_{\mathrm{source,pupil}},
E_{\mathrm{source,pupil}}
\right\rangle.
$$

The coherent modal coupling efficiency is therefore:

$$
\boxed{
\eta_{\mathrm{coherent}}
=
\frac{
|a_{\mathrm{coherent}}|^2
}{
P_{\mathrm{coherent,pupil}}
}
}
$$

assuming the fibre mode is normalised to unit power.

Unlike the incoherent case, interference between different source positions is retained. Even though we assume zero intrinsic source phase differences, the geometrical pupil tilts still produce spatially varying complex phases.

### 7. Variable Names

| Variable | Description | Shape / Type |
|---|---|---|
| `source_N` | Number of angular source samples along each axis | Integer |
| `source_sigma` | Gaussian standard deviation in $\lambda/D$ units | Scalar |
| `QX`, `QY` | Angular coordinates of the source in $\lambda/D$ units | `(source_N, source_N)` |
| `I_source_sky` | Normalised Gaussian surface brightness / fractional flux | `(source_N, source_N)` |
| `PX`, `PY` | Physical telescope pupil coordinates in metres | `(n_grid, n_grid)` |
| `D_M1` | VLT primary mirror diameter in metres | Scalar |
| `E_base_pupil` | Normalised on-axis telescope pupil electric field | Complex 2D array |
| `E_m_pupil` | Normalised back-propagated fibre-mode electric field | Complex 2D array |
| `tilt` | Angular-position-dependent pupil phase ramp | Complex 2D array |
| `E_psf_pupil` | Point-source pupil electric field at a given angular position | Complex 2D array |
| `amplitude` | Complex overlap of the point-source field with the fibre mode | Complex scalar |
| `powers` | Point-source modal coupling efficiency | Real scalar |
| `weight` | Square root of fractional source brightness | Real scalar |
| `source_coupling` | Point-source coupling efficiency at every source position | `(source_N, source_N)` |
| `incoherent_modal_power` | Brightness-weighted modal power contribution per source position | `(source_N, source_N)` |
| `coherent_amplitude` | Sum of complex modal coupling amplitudes | Complex scalar |
| `E_source_pupil` | Total coherent extended-source pupil electric field | Complex 2D array |
| `P_coherent_pupil` | Total coherent incident power across the pupil | Real scalar |
| `eta_incoherent` | Incoherent extended-source coupling efficiency | Real scalar |
| `eta_coherent` | Coherent extended-source coupling efficiency | Real scalar |

### 8. Numerical Assumptions and Next Steps

- The source is represented by discrete Gaussian-weighted point-source elements.
- All coherent source elements have zero relative intrinsic phase.
- The VLT pupil is diffraction-limited, with no atmospheric wavefront aberrations.
- `E_base_pupil` and `E_m_pupil` are normalised to unit power.
- The pupil and fibre front-face grids must form a consistent Fourier-transform pair.
- The back-propagated fibre mode is not artificially truncated at the telescope aperture.
- The coupling efficiency is calculated for one guided fibre mode; additional supported LP modes can be included later.
- The source angular sampling and extent should be tested for numerical convergence.

**Next step:** After reproducing the Gaussian calculation, replace `I_source_sky` with a Sérsic surface brightness distribution. The pupil phase tilts, coherent and incoherent summations, and modal overlap calculation can remain unchanged.
